In [78]:
import os
import torch
import torchaudio
import random
import torch.nn as nn
import torch.nn.functional as F
import librosa
from torch.utils.data import Dataset, DataLoader, random_split

In [80]:
class SpeechCommandsDataset(Dataset):
    def __init__(self, root_dir, keywords=None, sample_rate=16000,
                 noise_prob=0.8, snr_db=10, n_mfcc=40):
        self.root_dir = root_dir
        self.sample_rate = sample_rate
        self.noise_prob = noise_prob
        self.snr_db = snr_db
        self.n_mfcc = n_mfcc

        self.keywords = keywords or sorted([
            d for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d)) and not d.startswith('_')
        ])
        self.label2idx = {label: idx for idx, label in enumerate(self.keywords)}

        self.samples = []
        for label in self.keywords:
            folder = os.path.join(root_dir, label)
            for file in os.listdir(folder):
                if file.endswith('.wav'):
                    self.samples.append((os.path.join(folder, file), label))

        self.noise_clips = []
        noise_path = os.path.join(root_dir,"_background_noise_")
        if os.path.exists(noise_path):
            for file in os.listdir(noise_path):
                if file.endswith(".wav"):
                    noise_waveform, sr = librosa.load(os.path.join(noise_path, file))
                    # Resample if needed
                    if sr != self.sample_rate:
                        noise_waveform = librosa.resample(y=noise_waveform, orig_sr=sr, target_sr=self.sample_rate)
                    noise_waveform = torch.tensor(noise_waveform)
                    self.noise_clips.append(noise_waveform)
    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        waveform, sr = librosa.load(path)
        waveform = torch.tensor(waveform).unsqueeze(0)  # Add channel dimension

        if sr != self.sample_rate:
            waveform_np = waveform.squeeze(0).numpy()
            waveform_np = librosa.resample(y=waveform_np, orig_sr=sr, target_sr=self.sample_rate)
            waveform = torch.tensor(waveform_np).unsqueeze(0)

        waveform = self.pad_or_trim(waveform, self.sample_rate)
        waveform = self.add_background_noise(waveform)
        waveform_np = waveform.squeeze(0).numpy()
        mfcc = librosa.feature.mfcc(
            y=waveform_np,
            sr=self.sample_rate,
            n_mfcc=self.n_mfcc,
            n_fft=1024,
            hop_length=256,
            n_mels=64
        )
        mfcc = torch.tensor(mfcc).clamp(min=-10, max=10)  # (n_mfcc, T)
        mfcc = mfcc.unsqueeze(0)  # Add channel dimension (1, n_mfcc, T)

        # Pad or trim MFCC to fixed length (e.g., 98 frames)
        fixed_length = 98
        T = mfcc.size(-1)
        if T < fixed_length:
            pad_size = fixed_length - T
            mfcc = F.pad(mfcc, (0, pad_size))
        else:
            mfcc = mfcc[:, :, :fixed_length]

        return mfcc, self.label2idx[label]

    def pad_or_trim(self, waveform, length):
        if waveform.size(0) < length:
            waveform = F.pad(waveform, (0, length - waveform.size(0)))
        return waveform[:length]    
    def add_background_noise(self, waveform):
        if len(self.noise_clips) == 0 or random.random() > self.noise_prob:
            return waveform
        noise = random.choice(self.noise_clips)
        if noise.size(0) < waveform.size(0):
            noise = noise.repeat((waveform.size(0) // noise.size(0)) + 1)
        noise = noise[:waveform.size(0)]

        audio_power = waveform.norm(p=2)
        noise_power = noise.norm(p=2)
        snr = 10 ** (self.snr_db / 20)
        scale = audio_power / (snr * noise_power)
        return waveform + scale * noise

In [ ]:
import torch.nn as nn
import torch

class SpeechCommandCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3),  
            nn.ReLU(),
            nn.MaxPool2d(2),             
            nn.Conv2d(16, 32, kernel_size=3), 
            nn.ReLU(),
            nn.MaxPool2d(2),                  
        )
        dummy_input = torch.zeros(1, 1, 40, 98) 
        with torch.no_grad():
            dummy_output = self.conv(dummy_input)
            flattened_size = dummy_output.view(1, -1).shape[1]

        self.fc = nn.Sequential(
            nn.Linear(flattened_size, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)


In [82]:
def train_one_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = loss_fn(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            pred = out.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return correct / total


In [ ]:

def main():
    root = "C:\\Users\\Administrator\\Downloads\\speech_commands_v0.01" 
    keywords = ['yes', 'no','up', 'down', 'left', 'right']
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    dataset = SpeechCommandsDataset(root_dir=root, keywords=keywords)
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_ds, val_ds = random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=32)

    model = SpeechCommandCNN(num_classes=len(keywords)).to(device)
    #print(model)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(15):
        loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
        acc = evaluate(model, val_loader, device)
        print(f"Epoch {epoch+1}: Loss = {loss:.4f}, Val Acc = {acc:.4f}")

    torch.save(model.state_dict(), "speech_command_mfcc.pth")
    print(" Training finished and model saved.")

if __name__ == "__main__":
    main()


Epoch 1: Loss = 1.3308, Val Acc = 0.7006
Epoch 2: Loss = 0.5452, Val Acc = 0.7980
Epoch 3: Loss = 0.3401, Val Acc = 0.8586
Epoch 4: Loss = 0.2448, Val Acc = 0.8776
Epoch 5: Loss = 0.1700, Val Acc = 0.8723
Epoch 6: Loss = 0.1417, Val Acc = 0.8754
Epoch 7: Loss = 0.1080, Val Acc = 0.8786
Epoch 8: Loss = 0.0876, Val Acc = 0.8761
Epoch 9: Loss = 0.0859, Val Acc = 0.8642
Epoch 10: Loss = 0.0736, Val Acc = 0.8846
Epoch 11: Loss = 0.0706, Val Acc = 0.8944
Epoch 12: Loss = 0.0547, Val Acc = 0.8863
Epoch 13: Loss = 0.0593, Val Acc = 0.8691
Epoch 14: Loss = 0.0485, Val Acc = 0.8825
Epoch 15: Loss = 0.0538, Val Acc = 0.8832
✅ Training finished and model saved.
